In [1]:
import pyspark, os, sys, pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F 
import pyspark.sql.types as T
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("spark-sql-drill").getOrCreate()
sc = spark.sparkContext
print("Spark version:", spark.version)
spark

Python /usr/local/bin/python
Java /usr/lib/jvm/java-17-openjdk-arm64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 14:30:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2


# Drill

- display courses and their current instructors
- display courses and their historical instructors. Hint: use the function array_contains

## Dataset: University Courses and Instructors

Two JSON files represent courses and instructors at a university:

**`data/courses.json`** -- each course has:
- `cid`: course id
- `title`, `major`, `number`: metadata
- `currentInstructor`: id of the instructor *currently* teaching it

**`data/instructors.json`** -- each instructor has:
- `id`, `name`: identity
- `courses`: array of course ids they have taught *historically*

Key notes:
- Use `currentInstructor == id` for the current instructor join
- Use `array_contains(courses, cid)` to find which instructor has taught a course historically
- An outer join is needed because some courses have no current instructor assigned

In [2]:
coursesDF = spark.read.option("inferSchema", "true").json("data/courses.json");
instructorsDF = spark.read.option("inferSchema", "true").json("data/instructors.json");
coursesDF.printSchema()
instructorsDF.printSchema()

root
 |-- cid: long (nullable = true)
 |-- currentInstructor: long (nullable = true)
 |-- major: string (nullable = true)
 |-- number: string (nullable = true)
 |-- title: string (nullable = true)

root
 |-- courses: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)



In [3]:
spark.stop()